# 03. Kernel and model-form comparison

**Purpose.** Put the twelve GC-GP models from the `02_Model*` notebooks side by
side, so we can see how much the choice of model form, kernel and optimizer
matters for each property.

**What this notebook produces** (PNG and PDF figures at 600 dpi)

* One bar chart per property with test R², MAE and RMSE for every model.
* A combined figure with all properties (`Output_Data/kernel_comparison/All_Properties_Kernel_Comparison.png`).
* The underlying numbers as a CSV table.

**Inputs needed.** Run the twelve `02_Model*` notebooks first. Each one writes a
`<property>_metrics.csv` file that is read here.

**Model labels.** The labels follow the model definitions in the group
discussion slides (slides 18 and 19):

| Label | Folder | Kernel | Optimizer | Other settings |
|:--|:--|:--|:--|:--|
| model_1 to model_4 | `model_1` … `model_4` | RQ | L-BFGS-B | model forms 1 to 4 |
| model_5 | `model_4-opt1` | RBF | L-BFGS-B | |
| model_6 | `model_4-opt2` | Matérn 1/2 | L-BFGS-B | |
| model_7 | `model_4-opt3` | Matérn 3/2 | L-BFGS-B | |
| model_8 | `model_4-opt4` | Matérn 5/2 | L-BFGS-B | |
| model_9 | `model_4-opt5` | Matérn 3/2 | BFGS | |
| model_10 | `model_4-opt6` | Matérn 1/2 | BFGS | |
| model_11 | `model_4-opt7` | Matérn 1/2, Matérn 3/2 | L-BFGS-B | anisotropic, noise fixed |
| model_12 | `model_4-opt8` | Matérn 1/2, Matérn 3/2 | L-BFGS-B | anisotropic, noise trained |

Models 5 to 12 all use the linear-mean form (model 4). Where two kernels are
listed, the kernel depends on the property; see the run plan in each
`02_Model*` notebook.

## 1. Environment and setup

In [ ]:
import sys
from pathlib import Path

# Find the repository root (the folder that contains src/gcgp) and make the
# shared code importable. This works whether Jupyter starts in notebooks/ or
# in the repository root.
repoRoot = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "gcgp").is_dir())
sys.path.insert(0, str(repoRoot / "src"))

import matplotlib.pyplot as plt
import pandas as pd

from gcgp.settings import describeSetup, outputFolder, setGlobalSeed
from gcgp.properties import PROPERTY_ORDER
from gcgp.plotting import modelColorMap, plotMetricBars, savePlot
from gcgp.reporting import MODEL_FOLDERS, loadKernelComparison

setGlobalSeed()
outputDir = outputFolder("kernel_comparison")
pd.set_option("display.precision", 3)
print(describeSetup())

## 2. Load the saved metrics

Every model was fitted once per property, on the same curated dataset and
the same train/test split, so the bars within one panel are directly
comparable. Each run is scored on a single 80/20 split (85/15 for Model 12).

In [ ]:
metricsDf, missing = loadKernelComparison(outputDir, PROPERTY_ORDER)
metricsDf.to_csv(outputDir / "kernel_comparison_metrics.csv", index=False)

print(f"Loaded {len(metricsDf)} property-model results.")
if missing:
    print(f"{len(missing)} runs not found (run the matching 02_Model notebook):")
    print(", ".join(f"{p}/{f}" for p, f in missing))

## 3. Test-set metrics by property and model

Rows are models, columns are properties. Higher R² is better; lower MAE and RMSE
are better. Errors are in the units of each property.

In [ ]:
modelRows = sorted(metricsDf["Model"].unique(), key=lambda m: int(m.split("_")[1]))
propertyCols = [p for p in PROPERTY_ORDER if p in set(metricsDf["Property"])]
metricsDf.pivot(index="Model", columns="Property", values="R2").reindex(index=modelRows, columns=propertyCols)

## 4. Comparison figure

Each row is one property and each column one metric. Every model keeps the same
colour in every panel. The spread of bar heights within a panel shows how
sensitive that property is to the model choice.

In [ ]:
modelOrder = [label for label in MODEL_FOLDERS.values() if label in set(metricsDf["Model"])]
colors = modelColorMap(modelOrder)
codes = [p for p in PROPERTY_ORDER if p in set(metricsDf["Property"])]
metricNames = ["R2", "MAE", "RMSE"]

fig, axes = plt.subplots(len(codes), len(metricNames), figsize=(15, 4 * len(codes)), squeeze=False)
for i, code in enumerate(codes):
    propertyDf = metricsDf[metricsDf["Property"] == code].set_index("Model").loc[
        [m for m in modelOrder if m in set(metricsDf.loc[metricsDf["Property"] == code, "Model"])]].reset_index()
    for j, metric in enumerate(metricNames):
        plotMetricBars(axes[i, j], propertyDf, metric, colors, code)
fig.tight_layout()
savePlot(fig, outputDir / "All_Properties_Kernel_Comparison.png")
plt.show()
plt.close(fig)

## 5. Best model for each property

The model with the highest test R² on this single split. Because each property
is scored on one random split only, small differences between models are not
meaningful; the final models are evaluated with 10-fold cross-validation in
`04_Final_GCGP_CrossValidation.ipynb`.

In [ ]:
best = metricsDf.loc[metricsDf.groupby("Property")["R2"].idxmax()].set_index("Property").loc[codes]
best[["Model", "Folder", "R2", "MAE", "RMSE"]]

## 6. Summary of the results

Written from the table above and saved as
`Output_Data/kernel_comparison/results_summary.md`.

In [ ]:
from gcgp.narrative import describeKernelComparison, showAndSave

showAndSave(describeKernelComparison(metricsDf, best), outputDir / "results_summary.md");